# Financial Document Analysis Agent

**CSC3173 Artificial Intelligence: Programming Assignment 02**

This notebook builds an agent that answers financial-analysis questions about **Apple Inc.**
using its annual reports (Form 10-K) for fiscal years **2023, 2024 and 2025**.

The design follows the Medical Document (WHO statistics) Agent from class: PDFs are split into
chunks, embedded and indexed with FAISS, and a LangGraph loop lets a planner choose between
`RETRIEVE`, `CALCULATE`, `REPORT` and `FINISH`. The health-specific parts were replaced with
financial ones: a table-aware retrieval tool for the 10-K net sales tables, a Python calculation
tool, and a report generator that writes a PDF with charts and page citations.

Notebook outline:

| Stage | Section |
|---|---|
| 0 | Agentic components (Medical agent vs Financial agent) |
| A | Document processing |
| B | Semantic retrieval |
| C | Financial retrieval tool |
| D | Financial calculation tool |
| E | Report generation tool |
| F | LangGraph agent (state, planner, nodes, routing) |
| G | Test questions Q1 to Q3 and PDF report |

## Setup
Mount Google Drive (on Colab) and set the project paths. The three 10-K PDFs go in `Finance_AI_Agent/data/` (or directly in `Finance_AI_Agent/`).

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Demo/AI/Finance_AI_Agent"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = PROJECT_DIR
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Data dir   :", DATA_DIR)
print("Output dir :", OUTPUT_DIR)

### Package installation

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate
!pip install -q langgraph
!pip install -q fpdf2

In [ ]:
import fitz  # PyMuPDF
import re, glob, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)